# Ch.5　総合ミニ演習　—　乳がん診断データで Ch.1〜4 を再現しよう

---

## このノートブックの目的

Ch.1〜4 で学んだ手順を、**新しいデータ**に対して自分の手で再現することで、  
「自分でもデータ分析ができる」という感覚を持ち帰る。

| ステップ | 内容 | 目安時間 |
|---------|------|---------|
| Step 1 | データを読み込んで形を確認する（Ch.1 の復習） | 5分 |
| Step 2 | EDA でデータを可視化する（Ch.2 の復習） | 10分 |
| Step 3 | RandomForest でモデルを学習させる（Ch.3 の復習） | 10分 |
| Step 4 | 混同行列・特徴量重要度で評価する（Ch.3 の復習） | 10分 |
| Step 5 | 考察・まとめ（自由記述） | 5分 |

---

## 使うデータ：乳がん診断データ（`load_breast_cancer()`）

| 項目 | 内容 |
|------|------|
| 件数 | 569件 |
| 特徴量 | 30種類（細胞核の大きさ・形状など） |
| ラベル | 0 = 悪性（Malignant）、1 = 良性（Benign） |
| 出典 | scikit-learn 内蔵サンプルデータ |

> ワインの品種（3クラス）と違い、今回は **2クラス分類** です。  
> 医療データなので「悪性を見逃すこと」がどれだけ問題か、考えながら取り組んでみてください。

---

> ⏱️ **40分で問3まで終われば十分です。問4は時間に余裕があれば挑戦してください。**

---

> **🤖 Copilot Chat の使い方**  
> ブラウザで https://copilot.microsoft.com を開き、以下のテンプレートで質問してください。  
> ```
> 【やりたいこと】〇〇したい
> 【使うライブラリ】scikit-learn 0.23.2、pandas、matplotlib
> 【データの形】569行 × 30列の DataFrame、ラベルは 0（悪性）/ 1（良性）
> 【環境】Python 3.8.6、Windows、JupyterLab
> 【わからない点】〇〇の書き方
> ```

---
## Step 0. ライブラリの読み込み

まず必要なライブラリをすべて読み込みます。  
Ch.3 と同じセットです。エラーが出たら Copilot Chat に貼り付けて確認してください。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

print('ライブラリの読み込み完了')

---
## Step 1. データを読み込んで形を確認する ★☆☆

`load_breast_cancer()` でデータを取得し、DataFrame に変換します。  
Ch.1 で `load_wine()` に対してやったことを思い出しながら進めてください。

### 問1-1 ★☆☆　データを DataFrame に変換する

`load_wine()` の時と同じ手順で、乳がんデータを DataFrame に変換してください。  
末尾に `'診断'` 列（0 = 悪性、1 = 良性）を追加してください。

In [ ]:
cancer = load_breast_cancer()

# DataFrame に変換
df = pd.DataFrame(___, columns=___)   # ← 特徴量の数値 と 列名 を入れる
df['診断'] = ___                       # ← 正解ラベル（0 or 1）を追加

print('データ形状:', df.shape)
df.head()

### 問1-2 ★☆☆　データの基本情報を確認する

以下の3つを実行して、データの形・欠損値・統計量を確認してください。

In [ ]:
# (1) 列名・データ型・欠損値の確認
df.___()

In [ ]:
# (2) 基本統計量（平均・最大・最小など）
df.___()

In [ ]:
# (3) 各ラベルの件数（悪性 / 良性の内訳）
df['診断'].___()

> **確認ポイント**  
> - 欠損値はありますか？  
> - 悪性（0）と良性（1）の件数は均等ですか？

---
## Step 2. EDA でデータを可視化する ★★☆

重要な特徴量をグラフにして、悪性と良性の違いを目で確認します。

### 問2-1 ★☆☆　ヒストグラムで分布を見る

`mean radius`（細胞核の平均半径）のヒストグラムを、  
**悪性・良性で色を分けて**重ねて表示してください。

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

# 悪性（診断 == 0）と良性（診断 == 1）に分けてヒストグラムを描く
df[df['診断'] == ___]['mean radius'].hist(
    ax=ax, bins=30, alpha=0.6, label='悪性（0）', color='tomato'
)
df[df['診断'] == ___]['mean radius'].hist(
    ax=ax, bins=30, alpha=0.6, label='良性（1）', color='steelblue'
)

ax.set_xlabel('mean radius')
ax.set_ylabel('件数')
ax.set_title('mean radius の分布：悪性 vs 良性')
ax.legend()
plt.tight_layout()
plt.show()

> **発問**: mean radius が大きいほど悪性になりやすそうですか？　目で見て答えてください。

### 問2-2 ★★☆　箱ひげ図で複数の特徴量を比較する

以下の4つの特徴量について、悪性・良性ごとに箱ひげ図を並べて表示してください。  
`seaborn` の `boxplot` を使うと列名を指定するだけで簡単に描けます。

In [ ]:
features_to_check = ['mean radius', 'mean texture', 'mean perimeter', 'mean area']

fig, axes = plt.subplots(1, 4, figsize=(14, 5))

for ax, feat in zip(axes, features_to_check):
    sns.boxplot(
        data=df,
        x=___,      # ← X 軸：診断ラベル（0/1）
        y=___,      # ← Y 軸：確認したい特徴量名
        ax=ax,
        palette=['tomato', 'steelblue']
    )
    ax.set_title(feat)
    ax.set_xlabel('診断（0=悪性, 1=良性）')

plt.suptitle('悪性 vs 良性　特徴量の比較', fontsize=13)
plt.tight_layout()
plt.show()

### 問2-3 ★★☆　相関ヒートマップで変数間の関係を見る

特徴量が多い（30列）ので、まず `mean ` で始まる 10列だけを抜き出して  
相関ヒートマップを描いてください。

In [ ]:
# 'mean ' で始まる列だけ抜き出す
mean_cols = [col for col in df.columns if col.startswith('mean')]
df_mean = df[mean_cols]

plt.figure(figsize=(10, 8))
sns.heatmap(
    df_mean.___(),       # ← 相関係数行列を計算するメソッド
    annot=True,
    fmt='.1f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5
)
plt.title('mean 系特徴量の相関ヒートマップ', fontsize=13)
plt.tight_layout()
plt.show()

---
## Step 3. RandomForest でモデルを学習させる ★★☆

Ch.3 で実施したのと同じ手順で、乳がん診断の分類モデルを作ります。

### 問3-1 ★☆☆　特徴量（X）とラベル（y）に分けて、訓練/テスト分割する

In [ ]:
# 特徴量とラベルに分ける
X = cancer.___    # ← 特徴量の数値（numpy array）
y = cancer.___    # ← ラベル（0 or 1）

# 訓練データ（70%）とテストデータ（30%）に分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=___,       # ← 30% をテスト用に確保
    random_state=42
)

print('訓練データ:', X_train.shape)
print('テストデータ:', X_test.shape)

### 問3-2 ★☆☆　モデルを学習させて予測する

In [ ]:
# モデルを作成して学習
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.___(X_train, y_train)   # ← 学習するメソッド

# テストデータで予測
y_pred = model.___(X_test)    # ← 予測するメソッド

print('学習・予測完了')
print('予測結果（最初の10件）:', y_pred[:10])
print('正解ラベル（最初の10件）:', y_test[:10])

---
## Step 4. 混同行列・特徴量重要度で評価する ★★☆

### 問4-1 ★☆☆　正解率を確認する

In [ ]:
acc = accuracy_score(___, ___)   # ← 正解ラベル, 予測ラベル の順に入れる
print(f'正解率: {acc:.1%}')

### 問4-2 ★★☆　混同行列を描く

Ch.3 と同じ手順で混同行列を描いてください。  
ただし今回はラベルが `['悪性（Malignant）', '良性（Benign）']` の2クラスです。

In [ ]:
cm = confusion_matrix(y_test, y_pred)
labels = ['悪性（Malignant）', '良性（Benign）']

cm_df = pd.DataFrame(
    cm,
    index=[f'実際: {n}' for n in ___],    # ← ラベルのリスト
    columns=[f'予測: {n}' for n in ___]   # ← ラベルのリスト
)

plt.figure(figsize=(6, 4))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', linewidths=0.5)
plt.title(f'混同行列　（正解率: {acc:.1%}）', fontsize=13)
plt.tight_layout()
plt.show()

print(cm_df)

> **考えてみよう**  
> 混同行列の4つのマスのうち、医療現場で最も避けるべき「誤り」はどれですか？  
> → 悪性を良性と判定してしまうこと（左下のマス）、  
>   または良性を悪性と判定してしまうこと（右上のマス）、どちらが問題でしょうか？

### 問4-3 ★★☆　特徴量重要度を可視化する

どの特徴量が診断に最も貢献したかを確認します。  
上位10位だけを棒グラフで表示してください。

In [ ]:
importances = model.feature_importances_
feat_df = pd.DataFrame({
    '特徴量': cancer.feature_names,
    '重要度': ___   # ← 重要度の配列を入れる
}).sort_values('重要度', ascending=False)

# 上位10件だけ表示
top10 = feat_df.head(10)

plt.figure(figsize=(8, 5))
plt.barh(top10['特徴量'], top10['重要度'], color='steelblue')
plt.xlabel('重要度')
plt.title('特徴量重要度 Top10（RandomForest）', fontsize=13)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print('上位3特徴量:')
print(feat_df.head(3).to_string(index=False))

> **Ch.2 の EDA との照らし合わせ**  
> Step 2 で「悪性と良性の差が大きかった特徴量」と、重要度の高い特徴量は一致していますか？

---
## Step 5. 考察・まとめ ★★★

コードを書く必要はありません。自分の言葉で答えてください。  
（正解はありません。気づいたことを書けばOKです）

### 問5-1　モデルの性能について

今回の正解率はどう評価しますか？  
医療現場で使うとしたら、この精度で十分だと思いますか？

In [ ]:
# ↓ このセルに考察を日本語で書いてください（コードでなくコメントとして）

# 例）正解率は〇〇%で、特に悪性の見落とし（偽陰性）が〇件あった。
#     医療現場では偽陰性を減らすことが最優先なので…

### 問5-2　EDA とモデルの接続

Ch.2 で仮説を立てた変数と、Ch.3・Ch.4 で「重要度が高かった変数」は一致していましたか？  
EDA → モデル構築の流れを振り返ってみてください。

In [ ]:
# ↓ このセルに考察を日本語で書いてください

# 例）mean radius は EDA でも悪性と良性の差が大きく、特徴量重要度でも上位だった。
#     一方で〇〇は相関が高いと思ったが重要度は低かった。理由として…

### 問5-3 ★★★（発展）　悪性の見逃しを減らすには？

RandomForest の `predict_proba()` を使うと、各サンプルが「悪性である確率」を取得できます。  
デフォルトの判定閾値は 0.5 ですが、これを 0.3 に下げると何が変わるか試してみてください。

ヒント：Copilot Chat に以下を聞いてみてください。  
「`predict_proba` で確率を取得して、閾値を 0.3 に変えて予測結果を得るには？」

In [ ]:
# 発展問題：閾値を変えて偽陰性（悪性の見逃し）を減らす

# (1) 各サンプルが「良性（クラス1）」である確率を取得
proba = model.predict_proba(X_test)

# (2) 良性確率が 0.7 未満 → 悪性と判定（= 悪性確率 > 0.3 で悪性）
y_pred_strict = (proba[:, ___] < ___).astype(int)   # ← インデックスと閾値を入れる

# (3) 新しい混同行列
cm_strict = confusion_matrix(y_test, y_pred_strict)
acc_strict = accuracy_score(y_test, y_pred_strict)

cm_strict_df = pd.DataFrame(
    cm_strict,
    index=[f'実際: {n}' for n in labels],
    columns=[f'予測: {n}' for n in labels]
)

plt.figure(figsize=(6, 4))
sns.heatmap(cm_strict_df, annot=True, fmt='d', cmap='Oranges', linewidths=0.5)
plt.title(f'混同行列（閾値0.3）　正解率: {acc_strict:.1%}', fontsize=13)
plt.tight_layout()
plt.show()

print('\n閾値0.5 と 0.3 の比較:')
print(f'  閾値0.5 → 正解率: {acc:.1%}')
print(f'  閾値0.3 → 正解率: {acc_strict:.1%}')

---

## ⑥【発展】Precision / Recall を計算する ★★★

早く終わった方のみ取り組んでください。

> **正解率（Accuracy）だけでは見えないことがある**
>
> 医療診断では「悪性なのに良性と判定してしまう（見逃し）」が特に危険です。  
> Precision / Recall はこの「どんな間違いをしたか」を定量化します。

| 指標 | 意味 | 医療診断での重要性 |
|------|------|-----------------|
| **Precision** | 「悪性と予測した中で、本当に悪性の割合」 | 誤検知を減らしたいとき |
| **Recall** | 「実際の悪性のうち、正しく悪性と予測できた割合」 | **見逃しを減らしたいとき（重要）** |

**考えてみよう：** 正解率が 95% でも Recall が 60% だとしたら、医療現場でこのモデルを使えますか？

In [ ]:
from sklearn.metrics import precision_score, recall_score

precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)

print(f"正解率（Accuracy） : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision（陽性の精度）: {precision:.3f}  ← 悪性と予測した中で本当に悪性の割合")
print(f"Recall（見逃しのなさ）: {recall:.3f}  ← 実際の悪性のうち正しく検出できた割合")

# 考察：正解率が高くても Recall が低い場合、医療現場では危険なモデルになりうる
# あなたのモデルの Recall はどうでしたか？

---
## お疲れさまでした！

### 今日学んだことの総まとめ

| チャプター | やったこと | 使ったツール |
|-----------|----------|------------|
| Ch.1 | データ読み込み・集計・欠損値確認 | pandas |
| Ch.2 | ヒストグラム・箱ひげ図・相関ヒートマップ | matplotlib, seaborn |
| Ch.3 | 機械学習（RandomForest）で分類 | scikit-learn |
| Ch.4 | 時系列データの異常検知（線形回帰＋異常スコア） | scikit-learn, numpy |
| Ch.5 | 新しいデータで Ch.1〜4 を自分で再現 | 全部 |

---

### 持ち帰ってほしいこと

```
① データを見る　→　② 仮説を立てる　→　③ モデルで検証する　→　④ 結果を解釈する
```

この4ステップは、どんなデータでも同じです。  
今日使ったノートブックは、明日からの業務でそのままテンプレートとして使えます。

---

> **自分の業務データがあれば、ぜひ試してみてください。**  
> わからないことがあれば Copilot Chat に聞きながら進められます。